### Scratch book: 
#### SQL Queries for college hockey stat exploration

### Setup - Depend - Ect.

In [13]:
import os
import sys
from pathlib import Path
import pandas as pd

import numpy as np
import requests
from bs4 import BeautifulSoup
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import matplotlib.font_manager as fm
from matplotlib.font_manager import FontProperties
from matplotlib.offsetbox import OffsetImage
from matplotlib.ticker import PercentFormatter
from matplotlib.ticker import ScalarFormatter
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from PIL import Image

import numpy as np

# Optional advanced stats support
try:
    from statsmodels.nonparametric.smoothers_lowess import lowess as sm_lowess
    from statsmodels.regression.quantile_regression import QuantReg
    _HAS_STATSMODELS = True
except Exception:
    sm_lowess = None
    QuantReg = None
    _HAS_STATSMODELS = False


import config

# ======= BASE PATHS =======
try:
    # Works when running as a script
    base_dir = Path(__file__).resolve().parent
except NameError:
    # Fallback for notebooks or interactive mode
    base_dir = Path.cwd()

# one directory up (your config.py lives here)
config_folder = base_dir.parent

# two directories up (for TEMP, data, images)
project_root = base_dir.parent.parent



# ======= DATA FOLDERS =======
temp_folder = project_root / "TEMP"
data_folder = project_root / "data"
roster_folder = data_folder / "player_info"
school_info_folder = data_folder / "school_info"

# ======= IMAGE FOLDERS =======
img_folder = project_root / "images"
logo_folder = img_folder / "logos"
background_folder = img_folder / "background"
plot_folder = project_root / "TEMP" / "IMAGE" / "scatter_plots"

# ======= IMPORT CONFIG =======
sys.path.insert(0, str(config_folder))
import config  # now you can import config.py

# ======= LOAD DATA =======
roster_file = roster_folder / "roster_10_30_25.csv"
roster_df = pd.read_csv(roster_file)
roster_df["Current Team"] = roster_df["Current Team"].replace("RPI", "Rensselaer")

print(roster_df.columns)

school_info_file = school_info_folder / "arena_school_info.csv"
school_info_df = pd.read_csv(school_info_file)

# Check the Config import
# print((config_folder / "config.py").read_text())

Index(['Current Team', 'Last_Name', 'First_Name', 'No', 'Position', 'Yr', 'Ht',
       'Wt', 'DOB', 'Hometown', 'Height_Inches', 'Draft_Year', 'NHL_Team',
       'D_Round', 'Last Team', 'League', 'City', 'State_Province', 'Country'],
      dtype='object')


### Connect to database

In [14]:
## Connect to database using the recent_clean_db path from config.py
import sqlite3

### CONFIG FILE NOT WORKING AS EXPECTED - MANUAL FIX ####
data_folder = ('../../data/db/')
# filename = '2025_Feb_13_CLEAN.db'
filename = 'Season_YTD.db'
recent_clean_db = data_folder + filename
########### END MANUAL FIX ###########

conn = sqlite3.connect(recent_clean_db)
cursor = conn.cursor()
print("Connected to database:", config.recent_clean_db)


Connected to database: ../../data/db/Season_YTD.db


In [15]:
pen = pd.read_sql("SELECT * FROM penalty_summary;", conn)
goals = pd.read_sql("SELECT * FROM scoring_summary;", conn)
games = pd.read_sql("SELECT Game_ID, Home_Team, Away_Team FROM game_details;", conn)
lines = pd.read_sql("SELECT Game_ID, Team, goalsT FROM linescore;", conn)


# ----------------------------
# Team name normalization (drop-in)
# ----------------------------
TEAM_ALIAS = {
    "Rensselaer": "RPI",
    "St. Thomas": "St Thomas",
    "St. Cloud State": "St Cloud State",
    "St. Lawrence": "St Lawrence",
    # add more as you bump into them
}

def canon_team(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip()
    return TEAM_ALIAS.get(s, s)

# Apply to all team-name columns we use
for df, cols in [
    (pen,   ["Team"]),                 # <- if your penalty table uses a different column name, change it here
    (goals, ["Team"]),
    (games, ["Home_Team", "Away_Team"]),
    (lines, ["Team"]),
]:
    for c in cols:
        if c in df.columns:
            df[c] = df[c].map(canon_team)


# ----------------------------
# Helpers: time parsing (supports 1/2/3, and optionally OT if present)
# ----------------------------
def period_start_and_len(period):
    p = str(period).strip().lower()
    # Common variants: "1st", "1", "2nd", "3", "OT", "Overtime", etc.
    if p.startswith("1"): return 0, 1200
    if p.startswith("2"): return 1200, 1200
    if p.startswith("3"): return 2400, 1200
    if "ot" in p or "over" in p: return 3600, 300   # if you want OT
    return None, None

def mmss_to_sec(t):
    if pd.isna(t): return None
    s = str(t).strip()
    if ":" not in s: return None
    m, sec = s.split(":")
    try:
        return int(m) * 60 + int(sec)
    except:
        return None

def abs_time(period, time_str, include_ot=True):
    base, plen = period_start_and_len(period)
    if base is None: 
        return None
    if (base == 3600) and (not include_ot):
        return None
    sec = mmss_to_sec(time_str)
    if sec is None:
        return None
    # Treat "20:00" / "5:00" as horn, not a second before
    if plen is not None and sec == plen:
        return base + plen
    return base + sec

# ----------------------------
# Penalty typing (cancellable vs not)
# ----------------------------
def penalty_seconds(pen_min):
    # Pen_Length stored as TEXT in your DB
    try:
        return int(float(pen_min)) * 60
    except:
        return None

def is_cancellable(pen_min):
    # Majors do not cancel; minors & double minors do.
    # If you later add misconducts etc., you can refine this.
    try:
        m = int(float(pen_min))
    except:
        return False
    return m in (2, 4)

# ----------------------------
# Event-driven strength simulator for ONE game
# ----------------------------
def simulate_strength_game(game_id, home, away, pen_g, goals_g, include_ot=True):
    """
    Returns:
      intervals: list of dicts with time spans and skater counts
      events: list of dicts with goal/penalty events (optional debugging)
    """
    # Build penalty-start events
    pen_events = []
    for _, r in pen_g.iterrows():
        t0 = abs_time(r["Period"], r["Time"], include_ot=include_ot)
        dur = penalty_seconds(r["Pen_Length"])
        if t0 is None or dur is None or dur <= 0:
            continue
        team = r["Team"]
        pen_events.append({
            "t": t0,
            "type": "PEN_START",
            "team": team,
            "dur": dur,
            "cancellable": is_cancellable(r["Pen_Length"]),
        })

    # Build goal events
    goal_events = []
    for _, r in goals_g.iterrows():
        tg = abs_time(r["Period"], r["Time"], include_ot=include_ot)
        if tg is None:
            continue
        goal_events.append({
            "t": tg,
            "type": "GOAL",
            "team": r["Team"],
        })

    # Sort events by time; if same second, process penalties before goals
    def sort_key(e):
        pri = 0 if e["type"] == "PEN_START" else 1
        return (e["t"], pri)
    events = sorted(pen_events + goal_events, key=sort_key)

    # State: per team penalty queues
    # running penalties count down; queued penalties wait until a slot opens
    state = {
        home: {"running": [], "queued": []},
        away: {"running": [], "queued": []},
    }

    def running_count(team):
        return len(state[team]["running"])

    def skaters(team):
        # max 2 skaters down
        return 5 - min(2, running_count(team))

    def start_penalty(team, dur, cancellable):
        # if already 2 running, queue it; else start running immediately
        pen_obj = {"rem": dur, "cancellable": cancellable}
        if len(state[team]["running"]) < 2:
            state[team]["running"].append(pen_obj)
        else:
            state[team]["queued"].append(pen_obj)

    def tick(dt):
        # advance time by dt seconds, reduce remaining on running penalties
        for team in (home, away):
            for p in state[team]["running"]:
                p["rem"] -= dt

    def pop_expired_and_promote():
        # remove expired running; promote queued into running as slots open
        for team in (home, away):
            # remove expired
            state[team]["running"] = [p for p in state[team]["running"] if p["rem"] > 0]
            # promote queued into open slots
            while len(state[team]["running"]) < 2 and state[team]["queued"]:
                state[team]["running"].append(state[team]["queued"].pop(0))

    def next_expiration_time(now):
        # time until next running penalty expires
        times = []
        for team in (home, away):
            for p in state[team]["running"]:
                times.append(p["rem"])
        if not times:
            return None
        dt_min = min(times)
        return now + max(0, dt_min)

    def cancel_one_minor(scored_on_team):
        """
        When PP team scores, cancel ONE cancellable running penalty on the scored-on team.
        Rule proxy: cancel the cancellable penalty with the LEAST time remaining among RUNNING.
        (This matches standard bookkeeping for most play-by-play reconstructions.)
        """
        cancellables = [p for p in state[scored_on_team]["running"] if p["cancellable"]]
        if not cancellables:
            return False
        # pick the one with smallest remaining
        pmin = min(cancellables, key=lambda p: p["rem"])
        state[scored_on_team]["running"].remove(pmin)
        # promote queued if any
        while len(state[scored_on_team]["running"]) < 2 and state[scored_on_team]["queued"]:
            state[scored_on_team]["running"].append(state[scored_on_team]["queued"].pop(0))
        return True

    # Simulation loop
    intervals = []
    debug_events = []

    now = 0
    end_of_game = 3600 + (300 if include_ot else 0)

    i = 0
    while now <= end_of_game:
        next_event_t = events[i]["t"] if i < len(events) else None
        next_exp_t = next_expiration_time(now)

        # choose next time to jump to
        candidates = [t for t in [next_event_t, next_exp_t, end_of_game] if t is not None]
        t_next = min(candidates) if candidates else end_of_game

        if t_next > now:
            # record interval with current skater counts
            intervals.append({
                "Game_ID": game_id,
                "t_start": now,
                "t_end": t_next,
                "home": home,
                "away": away,
                "home_skaters": skaters(home),
                "away_skaters": skaters(away),
            })
            tick(t_next - now)
            now = t_next
            pop_expired_and_promote()

        # process all events at this second (pen starts then goals due to sorting)
        while i < len(events) and events[i]["t"] == now:
            e = events[i]
            if e["type"] == "PEN_START":
                start_penalty(e["team"], e["dur"], e["cancellable"])
                debug_events.append({**e, "Game_ID": game_id})
                pop_expired_and_promote()

            elif e["type"] == "GOAL":
                scoring_team = e["team"]
                other_team = home if scoring_team == away else away

                # Determine if this goal was scored while scoring team had a manpower advantage
                # If yes, cancel one minor on the scored-on team (if any cancellable running)
                adv = (skaters(scoring_team) > skaters(other_team))
                cancelled = False
                if adv:
                    cancelled = cancel_one_minor(other_team)

                debug_events.append({
                    **e,
                    "Game_ID": game_id,
                    "scoring_team": scoring_team,
                    "other_team": other_team,
                    "home_skaters": skaters(home),
                    "away_skaters": skaters(away),
                    "advantaged_goal": adv,
                    "cancelled_minor": cancelled,
                })
                pop_expired_and_promote()

            i += 1

        if now == end_of_game:
            break

    return intervals, debug_events

# ----------------------------
# Build 5v3 opportunities across all games
# ----------------------------
pen["t_abs"] = pen.apply(lambda r: abs_time(r["Period"], r["Time"], include_ot=True), axis=1)
goals["t_abs"] = goals.apply(lambda r: abs_time(r["Period"], r["Time"], include_ot=True), axis=1)

# Keep only rows with usable time
pen2 = pen[pen["t_abs"].notna()].copy()
goals2 = goals[goals["t_abs"].notna()].copy()

opps = []

for _, gr in games.iterrows():
    gid = gr["Game_ID"]
    home = gr["Home_Team"]
    away = gr["Away_Team"]

    pen_g = pen2[pen2["Game_ID"] == gid]
    goals_g = goals2[goals2["Game_ID"] == gid]

    intervals, dbg = simulate_strength_game(gid, home, away, pen_g, goals_g, include_ot=True)
    int_df = pd.DataFrame(intervals)
    if int_df.empty:
        continue

    # 5v3 for home: home has 5, away has 3
    int_df["home_5v3"] = (int_df["home_skaters"] == 5) & (int_df["away_skaters"] == 3)
    int_df["away_5v3"] = (int_df["away_skaters"] == 5) & (int_df["home_skaters"] == 3)

    # Helper: pull goals in a time window
    gg = goals_g.copy()
    gg["t_abs"] = gg["t_abs"].astype(int)

    def goals_in_window(team, t0, t1):
        return gg[(gg["Team"] == team) & (gg["t_abs"] >= t0) & (gg["t_abs"] < t1)].shape[0]

    # Identify continuous spans (opportunities)
    def add_spans(mask_col, advantaged_team, sh_team):
        sub = int_df[int_df[mask_col]].copy()
        if sub.empty:
            return
        sub = sub.sort_values("t_start")

        # merge contiguous/adjacent segments
        cur_s, cur_e = None, None
        for _, r in sub.iterrows():
            s, e = int(r["t_start"]), int(r["t_end"])
            if cur_s is None:
                cur_s, cur_e = s, e
            elif s <= cur_e:  # overlap/adjacent
                cur_e = max(cur_e, e)
            else:
                # finalize current span
                gf = goals_in_window(advantaged_team, cur_s, cur_e)
                ga = goals_in_window(sh_team, cur_s, cur_e)  # SH goals allowed during 5v3
                opps.append({
                    "Game_ID": gid,
                    "Adv_Team": advantaged_team,
                    "SH_Team": sh_team,
                    "t_start": cur_s,
                    "t_end": cur_e,
                    "duration_sec": cur_e - cur_s,
                    "GF_while_5v3": gf,
                    "GA_while_5v3": ga,
                    "Scored_on_5v3": gf > 0,
                    "Allowed_SH_on_5v3": ga > 0,
                })
                cur_s, cur_e = s, e

        # finalize last span
        gf = goals_in_window(advantaged_team, cur_s, cur_e)
        ga = goals_in_window(sh_team, cur_s, cur_e)
        opps.append({
            "Game_ID": gid,
            "Adv_Team": advantaged_team,
            "SH_Team": sh_team,
            "t_start": cur_s,
            "t_end": cur_e,
            "duration_sec": cur_e - cur_s,
            "GF_while_5v3": gf,
            "GA_while_5v3": ga,
            "Scored_on_5v3": gf > 0,
            "Allowed_SH_on_5v3": ga > 0,
        })

    add_spans("home_5v3", home, away)
    add_spans("away_5v3", away, home)

opps_53_df = pd.DataFrame(opps)

# ----------------------------
# Attach final W/L/T for advantaged team in each opportunity
# ----------------------------
lines["goalsT"] = pd.to_numeric(lines["goalsT"], errors="coerce")
score = lines[["Game_ID", "Team", "goalsT"]].copy()

opps_53_df = opps_53_df.merge(score, left_on=["Game_ID", "Adv_Team"], right_on=["Game_ID", "Team"], how="left") \
                       .rename(columns={"goalsT": "Adv_FinalGoals"}).drop(columns=["Team"], errors="ignore")

# opponent final goals
opps_53_df = opps_53_df.merge(score, left_on=["Game_ID", "SH_Team"], right_on=["Game_ID", "Team"], how="left") \
                       .rename(columns={"goalsT": "Opp_FinalGoals"}).drop(columns=["Team"], errors="ignore")

def wlt(a, b):
    if pd.isna(a) or pd.isna(b): return None
    if a > b: return "W"
    if a < b: return "L"
    return "T"

opps_53_df["Game_Result_for_Adv"] = opps_53_df.apply(lambda r: wlt(r["Adv_FinalGoals"], r["Opp_FinalGoals"]), axis=1)

# ----------------------------
# Team summary table (success + edge cases + outcomes)
# ----------------------------
if opps_53_df.empty:
    team_53_summary = pd.DataFrame()
else:
    g = opps_53_df.groupby("Adv_Team")

    team_53_summary = pd.DataFrame({
        "Opps_5v3": g.size(),
        "Opps_Scored": g["Scored_on_5v3"].sum(),
        "Opps_NoGoal": (g["Scored_on_5v3"].size() - g["Scored_on_5v3"].sum()),
        "ConvRate_anyGoal": g["Scored_on_5v3"].mean(),
        "GF_while_5v3": g["GF_while_5v3"].sum(),
        "GA_SH_while_5v3": g["GA_while_5v3"].sum(),
        "Opps_Allowed_SH": g["Allowed_SH_on_5v3"].sum(),
        "Avg_5v3_Duration_sec": g["duration_sec"].mean(),
        # “String two goals off a 5v3” proxy: scored during 5v3 AND had >=2 total goals
        # *during that same 5v3 span* (rare but catches oddities / same-second artifacts)
        "Opps_GF2plus_while_5v3": (opps_53_df["GF_while_5v3"] >= 2).groupby(opps_53_df["Adv_Team"]).sum(),
    }).fillna(0)

    # Game outcomes, split by whether they scored on the 5v3
    def wlt_counts(df):
        vc = df["Game_Result_for_Adv"].value_counts()
        return pd.Series({"W": int(vc.get("W", 0)), "L": int(vc.get("L", 0)), "T": int(vc.get("T", 0))})

    overall_wlt = opps_53_df.groupby("Adv_Team").apply(wlt_counts)
    scored_wlt = opps_53_df[opps_53_df["Scored_on_5v3"]].groupby("Adv_Team").apply(wlt_counts)
    noscore_wlt = opps_53_df[~opps_53_df["Scored_on_5v3"]].groupby("Adv_Team").apply(wlt_counts)

    # Join outcome splits
    overall_wlt.columns = [f"GameW_{c}" for c in overall_wlt.columns]  # GameW_W/GameW_L/GameW_T
    scored_wlt.columns = [f"ScoredW_{c}" for c in scored_wlt.columns]
    noscore_wlt.columns = [f"NoGoalW_{c}" for c in noscore_wlt.columns]

    team_53_summary = team_53_summary.join(overall_wlt, how="left") \
                                     .join(scored_wlt, how="left") \
                                     .join(noscore_wlt, how="left") \
                                     .fillna(0).astype({c: "int64" for c in team_53_summary.columns if c.endswith(("_W","_L","_T"))})

    # Sort by sample size then conversion
    team_53_summary = team_53_summary.sort_values(["Opps_5v3", "ConvRate_anyGoal"], ascending=[False, False])

# --- outputs ---
opps_53_df, team_53_summary


C:\Users\jbanc\AppData\Local\Temp\ipykernel_9000\574928993.py:406: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  overall_wlt = opps_53_df.groupby("Adv_Team").apply(wlt_counts)
C:\Users\jbanc\AppData\Local\Temp\ipykernel_9000\574928993.py:407: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  scored_wlt = opps_53_df[opps_53_df["Scored_on_5v3"]].groupby("Adv_Team").apply(wlt_counts)
C:\Users\jbanc\AppData\Local\T

(                                     Game_ID          Adv_Team        SH_Team  \
 0    2025-10-03-Connecticut-Colorado College  Colorado College    Connecticut   
 1         2025-10-03-Minnesota Duluth-Alaska  Minnesota Duluth         Alaska   
 2          2025-10-03-Merrimack-Mass. Lowell       Mass Lowell      Merrimack   
 3         2025-10-04-Lake Superior-Stonehill         Stonehill  Lake Superior   
 4         2025-10-04-Lake Superior-Stonehill         Stonehill  Lake Superior   
 ..                                       ...               ...            ...   
 368     2026-01-24-Augustana-Minnesota State   Minnesota State      Augustana   
 369               2026-01-24-Harvard-Cornell           Cornell        Harvard   
 370         2026-01-24-Clarkson-St. Lawrence       St Lawrence       Clarkson   
 371         2026-01-24-Clarkson-St. Lawrence       St Lawrence       Clarkson   
 372         2026-01-24-Clarkson-St. Lawrence       St Lawrence       Clarkson   
 
      t_start 

#### Question 1 - Team record after failing to convert on 5 minute PP, 5 on 3 PP

- Adam Nightengale asked this question during radio show - 1-26-26

In [16]:


# # ----------------------------
# # Load tables
# # ----------------------------
# pen = pd.read_sql("SELECT * FROM penalty_summary;", conn)
# goals = pd.read_sql("SELECT Game_ID, Period, Time, Team FROM scoring_summary;", conn)
# games = pd.read_sql("SELECT Game_ID, Home_Team, Away_Team FROM game_details;", conn)
# lines = pd.read_sql("SELECT Game_ID, Team, goalsT FROM linescore;", conn)


# # --- Find which column in penalty_summary is the penalized team ---
# # common possibilities I've seen across your tables / scrapes
# candidate_cols = [
#     "Team", "TEAM",
#     "Penalized_Team", "PenalizedTeam", "Pen_Team",
#     "Against", "Against_Team",
# ]

# pen_team_col = next((c for c in candidate_cols if c in pen.columns), None)
# if pen_team_col is None:
#     raise KeyError(f"Couldn't find penalized-team column in penalty_summary. Columns are: {list(pen.columns)}")

# # Standardize to a known name
# pen = pen.rename(columns={pen_team_col: "Penalized_Team"})

# # ----------------------------
# # Canonicalize team names (adjust as needed)
# # ----------------------------
# alias = {
#     "Rensselaer": "RPI",
#     "St. Thomas": "St Thomas",
#     "St. Cloud State": "St Cloud State",
#     "St. Lawrence": "St Lawrence",
# }

# def canon(t):
#     if pd.isna(t):
#         return None
#     t = str(t).strip()
#     return alias.get(t, t)

# for df, col in [
#     (pen, "Penalized_Team"),
#     (goals, "Team"),
#     (games, "Home_Team"),
#     (games, "Away_Team"),
#     (lines, "Team"),
# ]:
#     df[col] = df[col].map(canon)


# # ----------------------------
# # Time helpers (regulation only)
# # ----------------------------
# def period_start(period):
#     p = str(period).strip().lower()
#     if p.startswith("1"): return 0
#     if p.startswith("2"): return 1200
#     if p.startswith("3"): return 2400
#     return None  # ignore OT/SO

# def to_seconds(t):
#     if pd.isna(t) or ":" not in str(t):
#         return None
#     m, s = str(t).strip().split(":")
#     try:
#         return int(m) * 60 + int(s)
#     except:
#         return None

# def abs_time(period, time_str):
#     base = period_start(period)
#     sec = to_seconds(time_str)
#     if base is None or sec is None:
#         return None
#     return base + sec

# pen["pen_min"] = pd.to_numeric(pen["Pen_Length"], errors="coerce")
# pen["start_sec"] = pen.apply(lambda r: abs_time(r["Period"], r["Time"]), axis=1)
# pen["end_sec"] = pen["start_sec"] + pen["pen_min"] * 60

# goals["goal_sec"] = goals.apply(lambda r: abs_time(r["Period"], r["Time"]), axis=1)

# # ----------------------------
# # Identify 5-minute majors & PP team
# # ----------------------------
# majors = pen[
#     (pen["pen_min"] == 5) &
#     pen["start_sec"].notna() &
#     pen["end_sec"].notna()
# ].copy()

# majors = majors.merge(games, on="Game_ID", how="left")

# # penalized team = majors["Team"]
# # power-play team = opponent of penalized team
# majors["pp_team"] = np.where(
#     majors["Penalized_Team"] == majors["Home_Team"], majors["Away_Team"],
#     np.where(majors["Penalized_Team"] == majors["Away_Team"], majors["Home_Team"], None)
# )


# majors = majors[majors["pp_team"].notna()].copy()
# majors["major_id"] = np.arange(len(majors))

# # ----------------------------
# # Flag: did PP team score ANY goal during the 5:00 window?
# # (no reliance on goal strength labels)
# # ----------------------------
# g = goals[goals["goal_sec"].notna()].copy()

# tmp = g.merge(
#     majors[["major_id", "Game_ID", "pp_team", "start_sec", "end_sec"]],
#     left_on=["Game_ID", "Team"],
#     right_on=["Game_ID", "pp_team"],
#     how="inner"
# )

# tmp["goal_during_major"] = (
#     (tmp["goal_sec"] >= tmp["start_sec"]) &
#     (tmp["goal_sec"] < tmp["end_sec"])
# )

# scored_any = tmp.groupby("major_id")["goal_during_major"].any().rename("scored_during_major")
# majors = majors.join(scored_any, on="major_id")
# majors["scored_during_major"] = majors["scored_during_major"].fillna(False)

# # ----------------------------
# # Attach final score & compute W/L/T for the PP team
# # ----------------------------
# lines["goalsT"] = pd.to_numeric(lines["goalsT"], errors="coerce")

# majors = majors.merge(
#     lines, left_on=["Game_ID", "pp_team"], right_on=["Game_ID", "Team"], how="left"
# ).rename(columns={"goalsT": "pp_goals"}).drop(columns=["Team"], errors="ignore")

# majors["opp_team"] = np.where(
#     majors["pp_team"] == majors["Home_Team"], majors["Away_Team"], majors["Home_Team"]
# )

# majors = majors.merge(
#     lines, left_on=["Game_ID", "opp_team"], right_on=["Game_ID", "Team"], how="left"
# ).rename(columns={"goalsT": "opp_goals"}).drop(columns=["Team"], errors="ignore")

# def outcome(pp, opp):
#     if pd.isna(pp) or pd.isna(opp):
#         return None
#     if pp > opp: return "W"
#     if pp < opp: return "L"
#     return "T"

# majors["result"] = majors.apply(lambda r: outcome(r["pp_goals"], r["opp_goals"]), axis=1)

# # ----------------------------
# # "Reverse" set: majors where PP team DID score during the 5:00 window
# # ----------------------------
# scored_on_major = majors[(majors["scored_during_major"]) & (majors["result"].notna())].copy()

# # Team W-L-T in those situations (per major instance)
# team_wlt_scored = (
#     scored_on_major.groupby("pp_team")["result"]
#     .value_counts()
#     .unstack(fill_value=0)
# )

# for c in ["W", "L", "T"]:
#     if c not in team_wlt_scored.columns:
#         team_wlt_scored[c] = 0
# team_wlt_scored = team_wlt_scored[["W", "L", "T"]].copy()

# team_wlt_scored["Majors_ScoredOn"] = team_wlt_scored["W"] + team_wlt_scored["L"] + team_wlt_scored["T"]
# team_wlt_scored["WinPct_(ties_half)"] = (team_wlt_scored["W"] + 0.5 * team_wlt_scored["T"]) / team_wlt_scored["Majors_ScoredOn"]

# # ----------------------------
# # QC counts: total 5-min majors FOR and AGAINST each team (regardless of scoring)
# # ----------------------------
# majors_for = majors["pp_team"].value_counts().rename("Majors_For")          # opponent took major vs you
# majors_against = majors["Penalized_Team"].value_counts().rename("Majors_Against")    # you took major

# out = team_wlt_scored.join(majors_for, how="left").join(majors_against, how="left").fillna(0)
# out[["Majors_For", "Majors_Against"]] = out[["Majors_For", "Majors_Against"]].astype(int)

# # Optional extra QC: how often did they score during the major, out of majors_for?
# scored_rate = (scored_on_major["pp_team"].value_counts() / majors_for).rename("ScoreDuringMajor_Rate")
# out = out.join(scored_rate, how="left").fillna({"ScoreDuringMajor_Rate": 0})

# # Sort: most samples first
# out = out.sort_values(["Majors_ScoredOn", "WinPct_(ties_half)"], ascending=[False, False])

# out


In [17]:


# # ----------------------------
# # Load tables
# # ----------------------------
# pen = pd.read_sql("SELECT * FROM penalty_summary;", conn)
# goals = pd.read_sql("SELECT Game_ID, Period, Time, Team FROM scoring_summary;", conn)
# games = pd.read_sql("SELECT Game_ID, Home_Team, Away_Team FROM game_details;", conn)
# lines = pd.read_sql("SELECT Game_ID, Team, goalsT FROM linescore;", conn)

# # ----------------------------
# # Canonicalize team names
# # ----------------------------
# alias = {
#     "Rensselaer": "RPI",
#     "St. Thomas": "St Thomas",
#     "St. Cloud State": "St Cloud State",
#     "St. Lawrence": "St Lawrence",
# }

# def canon(t):
#     if pd.isna(t):
#         return None
#     t = str(t).strip()
#     return alias.get(t, t)

# for df, col in [
#     (pen, "Team"),
#     (goals, "Team"),
#     (games, "Home_Team"),
#     (games, "Away_Team"),
#     (lines, "Team"),
# ]:
#     df[col] = df[col].map(canon)

# # ----------------------------
# # Time helpers
# # ----------------------------
# def period_start(period):
#     p = str(period).lower()
#     if p.startswith("1"): return 0
#     if p.startswith("2"): return 1200
#     if p.startswith("3"): return 2400
#     return None  # ignore OT entirely

# def to_seconds(t):
#     if pd.isna(t) or ":" not in str(t):
#         return None
#     m, s = t.split(":")
#     return int(m) * 60 + int(s)

# def abs_time(row):
#     base = period_start(row["Period"])
#     sec = to_seconds(row["Time"])
#     if base is None or sec is None:
#         return None
#     return base + sec

# # ----------------------------
# # Build absolute timestamps
# # ----------------------------
# pen["start_sec"] = pen.apply(abs_time, axis=1)
# pen["pen_min"] = pd.to_numeric(pen["Pen_Length"], errors="coerce")
# pen["end_sec"] = pen["start_sec"] + pen["pen_min"] * 60

# goals["goal_sec"] = goals.apply(abs_time, axis=1)

# # ----------------------------
# # Filter to 5-minute majors
# # ----------------------------
# majors = pen[
#     (pen["pen_min"] == 5) &
#     pen["start_sec"].notna() &
#     pen["end_sec"].notna()
# ].copy()

# majors = majors.merge(games, on="Game_ID", how="left")

# # Power-play team = opponent of penalized team
# majors["pp_team"] = np.where(
#     majors["Team"] == majors["Home_Team"], majors["Away_Team"],
#     np.where(majors["Team"] == majors["Away_Team"], majors["Home_Team"], None)
# )

# majors["major_id"] = np.arange(len(majors))

# # ----------------------------
# # Join *all* goals by PP team
# # ----------------------------
# g = goals.merge(
#     majors[["major_id", "Game_ID", "pp_team", "start_sec", "end_sec"]],
#     left_on=["Game_ID", "Team"],
#     right_on=["Game_ID", "pp_team"],
#     how="inner"
# )

# # goal occurred during major window?
# g["goal_during_major"] = (
#     (g["goal_sec"] >= g["start_sec"]) &
#     (g["goal_sec"] < g["end_sec"])
# )

# scored = g.groupby("major_id")["goal_during_major"].any()

# majors = majors.join(scored, on="major_id")
# majors["scored_during_major"] = majors["goal_during_major"].fillna(False)

# # ----------------------------
# # Keep failures only
# # ----------------------------
# failed = majors[~majors["scored_during_major"]].copy()

# # ----------------------------
# # Attach final score and result
# # ----------------------------
# lines["goalsT"] = pd.to_numeric(lines["goalsT"], errors="coerce")

# failed = failed.merge(
#     lines, left_on=["Game_ID", "pp_team"], right_on=["Game_ID", "Team"],
#     how="left"
# ).rename(columns={"goalsT": "pp_goals"}).drop(columns=["Team"], errors="ignore")

# failed["opp_team"] = np.where(
#     failed["pp_team"] == failed["Home_Team"],
#     failed["Away_Team"],
#     failed["Home_Team"]
# )

# failed = failed.merge(
#     lines, left_on=["Game_ID", "opp_team"], right_on=["Game_ID", "Team"],
#     how="left"
# ).rename(columns={"goalsT": "opp_goals"}).drop(columns=["Team"], errors="ignore")

# def result(r):
#     if r["pp_goals"] > r["opp_goals"]: return "W"
#     if r["pp_goals"] < r["opp_goals"]: return "L"
#     return "T"

# failed["result"] = failed.apply(result, axis=1)

# ###
# # ----------------------------
# # Team-level record table
# # ----------------------------
# team_records = (
#     failed.groupby("pp_team")["result"]
#     .value_counts()
#     .unstack(fill_value=0)
# )

# # Ensure columns exist even if a result type is missing
# for c in ["W", "L", "T"]:
#     if c not in team_records.columns:
#         team_records[c] = 0

# team_records = team_records[["W", "L", "T"]].copy()

# # Add totals + a couple helpful rates
# team_records["Majors_Failed"] = team_records["W"] + team_records["L"] + team_records["T"]
# team_records["WinPct_(ties_half)"] = (team_records["W"] + 0.5 * team_records["T"]) / team_records["Majors_Failed"]

# # Optional: also count unique games (if multiple majors per game)
# team_records["Games_Failed"] = (
#     failed.groupby("pp_team")[["Game_ID"]]
#     .nunique()
#     .rename(columns={"Game_ID": "Games_Failed"})
# )

# # Sort: most majors first (or switch to WinPct if you want)
# team_records = team_records.sort_values(["Majors_Failed", "WinPct_(ties_half)"], ascending=[False, False])

# team_records.reset_index().rename(columns={"pp_team": "Team"}).round({"WinPct_(ties_half)": 3})

# team_records


# # ----------------------------
# # Summaries
# # ----------------------------
# per_major = failed["result"].value_counts().reindex(["W", "L", "T"]).fillna(0).astype(int)

# per_game = (
#     failed
#     .drop_duplicates(subset=["Game_ID", "pp_team"])
#     ["result"]
#     .value_counts()
#     .reindex(["W", "L", "T"])
#     .fillna(0)
#     .astype(int)
# )

# tmp = failed.merge(lines, left_on=["Game_ID", "pp_team"], right_on=["Game_ID", "Team"], how="left")
# print([c for c in tmp.columns if "Team" in c])


# print("Per major W-L-T:", f"{per_major['W']}-{per_major['L']}-{per_major['T']}", f"(N={len(failed)})")
# print("Per team-game W-L-T:", f"{per_game['W']}-{per_game['L']}-{per_game['T']}", f"(N={len(per_game)})")

# print(team_records.head(50))